# 12 — Freeze candidate B0–B4 cùng hard delta budget (CPU)

Notebook này chỉ chạy sau notebook 11 đã `AUDIT_COMPLETE 873/873`. Nó khóa đúng
corpus/audit vừa đo, đọc shot + mapping canonical và sinh bốn selector:

| Arm | Delta selector | Eligibility |
|---|---|---|
| B0 | Không delta — BTC-only | baseline |
| B1 | shot center | C0 |
| B2 | frame xa BTC nhất | C0/C1 |
| B3 | dynamic-density center/farthest | C0/C1 |
| B4 | uniform random, seed khóa | C0/C1 |

Hard delta budget là `floor(unique_BTC_count × 0.03)` và phải giống hệt cho
B1–B4. Underfill sẽ **BLOCK primary A/B**, không tự thu nhỏ budget và không lấy
candidate ngoài C0/C1. Exact BTC collision hoặc duplicate cũng làm run thất bại.

Notebook không tải MP4, không decode ảnh, không chạy GPU, không xem DEV/held-out
label và không ghi BTC/live index. Output là candidate manifests + worklist cho
lượt feature parity tiếp theo. Official Final Score vẫn `UNVERIFIED`.

### Chuẩn bị

1. Accelerator `None`, Internet bật.
2. Secrets: `GITHUB_TOKEN`, `MODAL_TOKEN_ID`, `MODAL_TOKEN_SECRET`.
3. Run All. Chỉ thành công nếu log cuối có `SELECTION_FROZEN` và equal budget PASS.

## 0 — Khóa đúng audit run vừa hoàn tất

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF = "feature/quang/storage"
REPO_DIR = Path("/kaggle/working/multimodal-fusion-engine")
WORK_DIR = Path("/kaggle/working/btc-b0-b4-candidate-freeze")

EXPECTED_VIDEO_COUNT = 873
EXPECTED_SHOT_COUNT = 97597
EXPECTED_AUDIT_MANIFEST_SHA256 = "370de9c53cdd84ee3c2bb5eb12e4a40c7d7d36676ba27f7dec0650d618d3586b"
EXPECTED_CORPUS_LOCK_HASH = "2e0887652464e0cc7942bedb94177cfe2dff5f7baf77081902461cf3c885d75f"
SELECTION_RUN_ID = "btc-b0-b4-selection-v1"
PUBLISH_RESULTS = True
CONFIRM_PUBLISH = "btc-b0-b4-selection-v1"

PRODUCER_CONFIG_REL = "config/keyframe_augmentation_kaggle.yaml"
AUDIT_CONFIG_REL = "config/keyframe_augmentation_v1.yaml"
AUDIT_SUBDIR = "gate1-audit-v1"
SELECTION_SUBDIR = "gate1-selection-v1"

if PUBLISH_RESULTS and CONFIRM_PUBLISH != SELECTION_RUN_ID:
    raise RuntimeError("Publish bị khóa: CONFIRM_PUBLISH phải bằng SELECTION_RUN_ID")
WORK_DIR.mkdir(parents=True, exist_ok=True)
print({"runtime": "CPU_ONLY", "input_videos": EXPECTED_VIDEO_COUNT, "heldout_accessed": False})

## 1 — Fetch checkout mới, reload code và ghi provenance

In [ ]:
import importlib
import json
import os
import subprocess
import sys
from datetime import datetime, timezone


def run(command: list[str], *, cwd: Path | None = None, env: dict[str, str] | None = None) -> str:
    result = subprocess.run(command, cwd=cwd, env=env, check=False, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(f"Command failed: {' '.join(command[:3])}\n{result.stderr[-1500:]}")
    return result.stdout.strip()


def refresh_repo() -> None:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    askpass = WORK_DIR / "git-askpass.sh"
    askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) printf x-access-token ;; *) printf %s "$GITHUB_TOKEN" ;; esac\n', encoding="utf-8")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update({"GITHUB_TOKEN": token, "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0"})
    try:
        if (REPO_DIR / ".git").is_dir():
            if run(["git", "status", "--porcelain"], cwd=REPO_DIR):
                raise RuntimeError("Checkout Kaggle có local changes; Restart Session rồi Run All")
            run(["git", "fetch", "--depth", "1", "origin", REPO_REF], cwd=REPO_DIR, env=env)
            run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR)
        else:
            run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], env=env)
    finally:
        askpass.unlink(missing_ok=True)


refresh_repo()
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "notebooks/requirements-kaggle-keyframe-augmentation.txt")])
sys.path.insert(0, str(REPO_DIR / "src"))

import yaml
from mfe.offline.bundle import BundleLayout
from mfe.offline.keyframe_augmentation.common import (
    atomic_write_json, atomic_write_jsonl, read_jsonl, sha256_file, sha256_json,
)
from mfe.offline.keyframe_augmentation.gate0 import read_and_validate_mapping
from mfe.offline.keyframe_augmentation.gate1 import (
    equalize_budget, generate_candidate_pools, load_shot_proposals,
)
from mfe.offline.keyframe_augmentation.models import AugmentationConfig
from mfe.offline.transport import ModalVolumeTransport

selection_freeze = importlib.import_module("mfe.offline.keyframe_augmentation.selection_freeze")
selection_freeze = importlib.reload(selection_freeze)
build_feature_worklist = selection_freeze.build_feature_worklist
crowding_diagnostics = selection_freeze.crowding_diagnostics
validate_frozen_selection = selection_freeze.validate_frozen_selection

REPO_COMMIT = run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR)
PRODUCER_CONFIG_PATH = REPO_DIR / PRODUCER_CONFIG_REL
AUDIT_CONFIG_PATH = REPO_DIR / AUDIT_CONFIG_REL
PRODUCER_CONFIG = yaml.safe_load(PRODUCER_CONFIG_PATH.read_text(encoding="utf-8"))
AUDIT_CONFIG = AugmentationConfig.model_validate(yaml.safe_load(AUDIT_CONFIG_PATH.read_text(encoding="utf-8")))
PRODUCER_CONFIG_SHA256 = sha256_file(PRODUCER_CONFIG_PATH)
AUDIT_CONFIG_SHA256 = sha256_file(AUDIT_CONFIG_PATH)
print({"repo_commit": REPO_COMMIT, "producer_config_sha256": PRODUCER_CONFIG_SHA256, "audit_config_sha256": AUDIT_CONFIG_SHA256, "seed": AUDIT_CONFIG.seed})

## 2 — Verify audit manifest và corpus lock từ notebook 11

In [ ]:
BTC = ModalVolumeTransport(PRODUCER_CONFIG["btc"]["volume"], create_if_missing=False)
RESEARCH = ModalVolumeTransport(PRODUCER_CONFIG["output"]["volume"], create_if_missing=False)
BTC_LAYOUT = BundleLayout(PRODUCER_CONFIG["btc"]["bundle"])
PRODUCER_ROOT = PRODUCER_CONFIG["output"]["root"].rstrip("/")
AUDIT_REMOTE_ROOT = f"{PRODUCER_ROOT}/{AUDIT_SUBDIR}"
SELECTION_REMOTE_ROOT = f"{PRODUCER_ROOT}/{SELECTION_SUBDIR}"

INPUT_DIR = WORK_DIR / "input"
OUTPUT_DIR = WORK_DIR / "output"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AUDIT_MANIFEST_PATH = INPUT_DIR / "audit-manifest.json"
RESEARCH.get(f"{AUDIT_REMOTE_ROOT}/audit-manifest.json", AUDIT_MANIFEST_PATH)
actual_audit_manifest_sha = sha256_file(AUDIT_MANIFEST_PATH)
if actual_audit_manifest_sha != EXPECTED_AUDIT_MANIFEST_SHA256:
    raise RuntimeError(f"Audit manifest drift: {actual_audit_manifest_sha}")
AUDIT_MANIFEST = json.loads(AUDIT_MANIFEST_PATH.read_text(encoding="utf-8"))
if AUDIT_MANIFEST["status"] != "AUDIT_COMPLETE" or AUDIT_MANIFEST["video_count"] != EXPECTED_VIDEO_COUNT:
    raise RuntimeError("Audit input chưa COMPLETE 873/873")
if AUDIT_MANIFEST["corpus_lock_hash"] != EXPECTED_CORPUS_LOCK_HASH:
    raise RuntimeError("Audit manifest trỏ sai corpus lock")

for name in ("corpus-lock.json", "per-video-audit.jsonl", "summary.json"):
    local = INPUT_DIR / name
    RESEARCH.get(f"{AUDIT_REMOTE_ROOT}/{name}", local)
    if sha256_file(local) != AUDIT_MANIFEST["files"][name]:
        raise RuntimeError(f"Audit artifact hash mismatch: {name}")

CORPUS_LOCK = json.loads((INPUT_DIR / "corpus-lock.json").read_text(encoding="utf-8"))
unsigned_lock = dict(CORPUS_LOCK)
lock_hash = unsigned_lock.pop("corpus_lock_hash")
if lock_hash != EXPECTED_CORPUS_LOCK_HASH or sha256_json(unsigned_lock) != lock_hash:
    raise RuntimeError("corpus-lock payload không hợp lệ")
AUDIT_SUMMARY = json.loads((INPUT_DIR / "summary.json").read_text(encoding="utf-8"))
if AUDIT_SUMMARY["shot_count"] != EXPECTED_SHOT_COUNT:
    raise RuntimeError("Shot count drift so với audit đã khóa")
PER_VIDEO_INPUT = list(read_jsonl(INPUT_DIR / "per-video-audit.jsonl"))
VIDEO_IDS = sorted(row["video_id"] for row in PER_VIDEO_INPUT)
if len(VIDEO_IDS) != EXPECTED_VIDEO_COUNT or len(VIDEO_IDS) != len(set(VIDEO_IDS)):
    raise RuntimeError("per-video audit không đủ 873 video disjoint")
PER_VIDEO_BY_ID = {row["video_id"]: row for row in PER_VIDEO_INPUT}
print({"audit_manifest_sha256": actual_audit_manifest_sha, "corpus_lock_hash": lock_hash, "videos": len(VIDEO_IDS), "shots": AUDIT_SUMMARY["shot_count"]})

## 3 — Đọc shot + mapping, xác minh hash và tạo pools canonical

In [ ]:
ARTIFACT_DIR = WORK_DIR / "artifacts"
PROPOSALS = []
MAPPINGS = {}
MAPPING_DUPLICATE_ALIASES = 0
for ordinal, video_id in enumerate(VIDEO_IDS, 1):
    audit_row = PER_VIDEO_BY_ID[video_id]
    shots_path = ARTIFACT_DIR / "shots" / f"{video_id}.jsonl"
    mapping_path = ARTIFACT_DIR / "mappings" / f"{video_id}.csv"
    RESEARCH.get(f"{PRODUCER_ROOT}/shots/{video_id}.jsonl", shots_path)
    BTC.get(BTC_LAYOUT.map_keyframes(video_id), mapping_path)
    if sha256_file(shots_path) != audit_row["shots_sha256"]:
        raise RuntimeError(f"{video_id}: shot hash drift")
    if sha256_file(mapping_path) != audit_row["btc_mapping_sha256"]:
        raise RuntimeError(f"{video_id}: mapping hash drift")
    video_proposals = load_shot_proposals(shots_path)
    if len(video_proposals) != audit_row["shot_count"]:
        raise RuntimeError(f"{video_id}: shot count drift")
    frames, mapping_report = read_and_validate_mapping(mapping_path, video_id=video_id)
    if len(frames) != audit_row["btc_unique_frames"]:
        raise RuntimeError(f"{video_id}: unique BTC count drift")
    PROPOSALS.extend(video_proposals)
    MAPPINGS[video_id] = frames
    MAPPING_DUPLICATE_ALIASES += len(mapping_report["duplicate_aliases"])
    if ordinal == 1 or ordinal % 25 == 0 or ordinal == EXPECTED_VIDEO_COUNT:
        print({"loaded_videos": ordinal, "expected": EXPECTED_VIDEO_COUNT, "shots": len(PROPOSALS), "unique_btc": sum(len(rows) for rows in MAPPINGS.values())}, flush=True)

if len(PROPOSALS) != EXPECTED_SHOT_COUNT:
    raise RuntimeError("Không selector trên partial shot corpus")
BTC_UNIQUE_COUNT = sum(len(rows) for rows in MAPPINGS.values())
AUDITS, POOLS = generate_candidate_pools(PROPOSALS, MAPPINGS, config=AUDIT_CONFIG)
SELECTED, BUDGET_REPORT = equalize_budget(
    POOLS, btc_count=BTC_UNIQUE_COUNT, guardrail_ratio=AUDIT_CONFIG.guardrail_ratio,
)
print({"btc_unique_count": BTC_UNIQUE_COUNT, "mapping_duplicate_aliases": MAPPING_DUPLICATE_ALIASES, "pool_sizes": BUDGET_REPORT["pool_sizes"], "budget": BUDGET_REPORT})

## 4 — Equal-budget proof, crowding diagnostic và feature worklist

In [ ]:
BUDGET_PROOF = validate_frozen_selection(
    selected=SELECTED, mappings=MAPPINGS, budget_report=BUDGET_REPORT,
)
FEATURE_WORKLIST = build_feature_worklist(SELECTED)
CROWDING = crowding_diagnostics(SELECTED)

for arm in ("B1", "B2", "B3", "B4"):
    atomic_write_jsonl(
        OUTPUT_DIR / f"candidate-{arm.lower()}.jsonl",
        (row.model_dump(mode="json") for row in SELECTED[arm]),
    )
atomic_write_jsonl(OUTPUT_DIR / "feature-worklist.jsonl", FEATURE_WORKLIST)
atomic_write_json(OUTPUT_DIR / "budget-proof.json", {**BUDGET_REPORT, **BUDGET_PROOF})
atomic_write_json(OUTPUT_DIR / "crowding-diagnostics.json", CROWDING)

FACTORIZED_PLAN = {
    "schema_version": 1,
    "status": "PLANNED_NOT_EXECUTED",
    "primary_ab": {
        "B0": "BTC_ONLY",
        "B1": "selector=shot_center; exposure/calibration=frozen",
        "B2": "selector=farthest; exposure/calibration=frozen",
        "B3": "selector=dynamic_density; exposure/calibration=frozen",
        "B4": "selector=uniform_random; exposure/calibration=frozen",
    },
    "ablations": {
        "E-S0": {"selector": "B3", "exposure": "frozen", "calibration": "frozen"},
        "E-X1": {"selector": "B3", "exposure": "origin_quota", "calibration": "frozen"},
        "E-C1": {"selector": "B3", "exposure": "frozen", "calibration": "origin_aware"},
        "E-XC": {"selector": "B3", "exposure": "origin_quota", "calibration": "origin_aware", "run_after_main_effects": True},
    },
    "feature_parity": {
        "status": "BLOCKED_PENDING_GPU_EXTRACTION",
        "required_same_as_btc": ["decoder", "RGB/BGR", "resize_crop", "interpolation", "model_weights_sha256", "preprocessing", "embedding_dimension", "normalization", "OCR_caption_schema"],
    },
    "O1-S": "BLOCKED_FEATURE_ARTIFACTS_REQUIRED",
    "O1-R": "BLOCKED_QUERY_RANK_TRACES_REQUIRED",
    "official_final_score": "UNVERIFIED",
    "heldout_accessed": False,
}
atomic_write_json(OUTPUT_DIR / "factorized-ablation-plan.json", FACTORIZED_PLAN)

SELECTION_SUMMARY = {
    "schema_version": 1,
    "status": "READY_FOR_FEATURE_EXTRACTION",
    "selection_run_id": SELECTION_RUN_ID,
    "input_corpus_lock_hash": EXPECTED_CORPUS_LOCK_HASH,
    "input_audit_manifest_sha256": EXPECTED_AUDIT_MANIFEST_SHA256,
    "repo_commit": REPO_COMMIT,
    "audit_config_sha256": AUDIT_CONFIG_SHA256,
    "seed": AUDIT_CONFIG.seed,
    "btc_unique_count": BTC_UNIQUE_COUNT,
    "locked_hard_delta_budget": BUDGET_PROOF["locked_hard_delta_budget"],
    "selected_counts": BUDGET_PROOF["selected_counts"],
    "unique_feature_work_items": len(FEATURE_WORKLIST),
    "feature_parity_status": "BLOCKED_PENDING_GPU_EXTRACTION",
    "official_final_score": "UNVERIFIED",
    "decision_authority": "candidate_freeze_only",
    "heldout_accessed": False,
    "btc_mutated": False,
}
atomic_write_json(OUTPUT_DIR / "selection-summary.json", SELECTION_SUMMARY)
print({"equal_budget_proof": BUDGET_PROOF, "selection_summary": SELECTION_SUMMARY})

## 5 — Publish research artifacts; freeze marker ghi cuối

In [ ]:
OUTPUT_FILES = sorted(path for path in OUTPUT_DIR.iterdir() if path.is_file())
OUTPUT_HASHES = {path.name: sha256_file(path) for path in OUTPUT_FILES}
SELECTION_FREEZE = {
    "schema_version": 1,
    "status": "SELECTION_FROZEN",
    "selection_run_id": SELECTION_RUN_ID,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "repo_commit": REPO_COMMIT,
    "input_audit_manifest_sha256": EXPECTED_AUDIT_MANIFEST_SHA256,
    "input_corpus_lock_hash": EXPECTED_CORPUS_LOCK_HASH,
    "config_sha256": AUDIT_CONFIG_SHA256,
    "seed": AUDIT_CONFIG.seed,
    "locked_hard_delta_budget": BUDGET_PROOF["locked_hard_delta_budget"],
    "selected_counts": BUDGET_PROOF["selected_counts"],
    "feature_work_items": len(FEATURE_WORKLIST),
    "output_root": SELECTION_REMOTE_ROOT,
    "files": OUTPUT_HASHES,
    "heldout_accessed": False,
    "btc_mutated": False,
}
SELECTION_FREEZE["freeze_payload_hash"] = sha256_json(SELECTION_FREEZE)
atomic_write_json(OUTPUT_DIR / "selection-freeze.json", SELECTION_FREEZE)

if PUBLISH_RESULTS:
    for path in OUTPUT_FILES:
        remote = f"{SELECTION_REMOTE_ROOT}/{path.name}"
        RESEARCH.put(path, remote)
        if RESEARCH.sha256(remote) != OUTPUT_HASHES[path.name]:
            raise RuntimeError(f"Remote verify failed: {remote}")
    freeze_path = OUTPUT_DIR / "selection-freeze.json"
    freeze_remote = f"{SELECTION_REMOTE_ROOT}/selection-freeze.json"
    RESEARCH.put(freeze_path, freeze_remote)
    if RESEARCH.sha256(freeze_remote) != sha256_file(freeze_path):
        raise RuntimeError("Remote selection freeze verify failed")

print("SELECTION_FROZEN")
print({"equal_budget": BUDGET_PROOF["selected_counts"], "hard_delta_budget": BUDGET_PROOF["locked_hard_delta_budget"], "feature_work_items": len(FEATURE_WORKLIST), "remote_output": SELECTION_REMOTE_ROOT if PUBLISH_RESULTS else "DISABLED", "freeze_sha256": sha256_file(OUTPUT_DIR / "selection-freeze.json")})

## 6 — Sau notebook 12

Không thêm các candidate vào BTC ngay. `feature-worklist.jsonl` là union decode
work để tránh chạy trùng frame xuất hiện ở nhiều arm; candidate manifest riêng
vẫn giữ membership chính xác của B1–B4.

Lượt tiếp theo phải tạo ảnh + feature bằng **cùng pipeline/fingerprint với BTC**,
ghi artifact research riêng và chạy feature-parity audit. Chỉ sau đó mới build
snapshot index riêng cho DEV A/B, đo O1-S/O1-R, crowding/rank exposure, official
Final Score và P50/P95. BTC-only production vẫn là default.